# Стратегии выбора актуальной записи ФП

В CRM одной карточке ФП (ИНН + дата выявления + номер + тип) может соответствовать
несколько строк — промежуточные состояния на этапе сценария и на этапе ИПУ.

**Задача:** определить стратегию, которая оставляет наиболее актуальную (финальную) запись.

Сценарии и ИПУ рассматриваются **раздельно**:

| Этап | Результат | Даты-маркеры |
|------|-----------|-------------|
| Сценарий | `DESC_TEXT_1` | `END_DATE_SCR_FCT`, `END_DATE_SCR_PLAN` |
| ИПУ | `DESC_TEXT` | `FIRST_END_DATE_EVENT`, `NEW_PLAN_END_DATE_EVT`, `END_EVENT_DATE_FACT` |

**Стратегии для каждого этапа:**
1. `keep='first'` — текущее поведение (базовая линия)
2. `keep='last'` — последняя строка в CSV
3. По самой поздней дате этапа
4. По ближайшей к `DATE_END_FP_SFP` дате этапа
5. Фильтрация: оставлять только записи с финальным результатом (+/-)

## 0. Конфигурация, загрузка, классификация

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

SCR_DATE_COLS = ["END_DATE_SCR_FCT", "END_DATE_SCR_PLAN"]
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT", "END_EVENT_DATE_FACT"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("OK")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})
    print("Переименована колонка DESC_TEXT.1 → DESC_TEXT_1")

print(f"Загружено: {len(raw):,} строк")

if "VAL" in raw.columns:
    raw = raw[raw["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам: {len(raw):,}")

raw["inn"] = raw["X_INN"].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(raw):,}")

raw["segment"] = raw["X_AREA_RESP"].str.strip().map(SEGMENT_MAP)
raw = raw[raw["segment"].notna()].copy()

raw["fp_num"]  = raw["NUMBER_FP_SFP"].str.strip()
raw["fp_type"] = raw["TYPE_FP"].str.strip()

# DESC_TEXT_1 = результат сценария, DESC_TEXT = результат ИПУ
# Нормализация: убираем неразрывные пробелы, схлопываем множественные пробелы
raw["scenario"] = raw["DESC_TEXT_1"].fillna("").str.replace(r'[\s\xa0]+', ' ', regex=True).str.strip()
raw["ipu"]      = raw["DESC_TEXT"].fillna("").str.replace(r'[\s\xa0]+', ' ', regex=True).str.strip()

# Парсинг дат
for col in SCR_DATE_COLS + IPU_DATE_COLS + ["DATE_END_FP_SFP"]:
    raw[f"_{col}_dt"] = pd.to_datetime(raw[col], dayfirst=True, errors="coerce")

print(f"Итого: {len(raw):,} строк, {raw['inn'].nunique():,} ИНН")
n_cards = raw.drop_duplicates(DEDUP_KEY).shape[0]
print(f"Уникальных карточек ФП: {n_cards:,}")

In [ ]:
# ── Классификация результатов сценария ──
SCR_POSITIVE = [
    "ФП/СФП устранен", "ФП/СФП урегулирован", "Микро_У ФП урегулирован",
    "МСБ_Принято решение УОБ о снятии ФП с контроля",
    "МСБ_Принято решение УЛ о снятии ФП с контроля",
    "1_ККБ/МСБ_ФП устранен", "СФП устранен",
    "Микро_У Требуется принятие решения УЛ о снятии фактора с контроля",
    "Принято решение УЛ о снятии ФП с контроля",
    "Микро_У Фактор устранен", "Денежная санкция уплачена",
    "ФП/СФП устранен (договор закрыт)", "Условие выполнено",
    "2_ККБ/МСБ_СФП устранен",
    "ДКБ_Принято решение УЛ о снятии ФП с контроля",
    "ДКБ_Принято решение УОБ об урегулировании ФП",
    "ДКБ_Принято решение УОБ о снятии ФП с контроля",
    "ФП устранен (договор закрыт)",
    "ДКБ_Не требуется решение УЛ/УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии фактора с контроля",
    "Микро_У ФП устранен", "ФП устранен",
    "ДКБ_ФП не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, возможно снятие ФП с контроля",
    "Принято решение УОБ о снятии ФП с контроля",
    "Микро_У СФП устранен",
    "ФП урегулирован, требуется принятие решения УОБ о снятии ФП с контроля",
    "ДКБ_СФП устранен",
    "Микро_У Принято решение УОБ о снятии ФП с контроля",
    "Микро_У СФП урегулирован",
    "Микро_У Принято решение УЛ о снятии ФП с контроля",
]

SCR_NEGATIVE = [
    "ФП/СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "3_ККБ/МСБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "ДКБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "12_ККБ/МСБ_ФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
    "ДКБ_ФП оказывает негативное влияние, требует рассмотрения УОБ вопроса о признании задолженности проблемной",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках сценария, оказывает негативное влияние, требует рассмотрения вопроса на УОБ об утверждении Плана устранения ФП",
    'ФП не устранен. Лимит снижен до "0". Лимит восстановлению не подлежит.',
    "Микро_У Техническое закрытие ФП в связи с запуском нового сценария по СФП (СФП 15.2У/15.2.1У)",
]

SCR_ERROR = [
    "0_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "0. Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "ДКБ_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "Микро_У Информация проверена, отсутствуют основания для выявления ФП/СФП",
]

SCR_NEUTRAL = [
    "ФП не устранен, не оказывает негативного влияния, требует рассмотрения вопроса на УОБ с целью снятия ФП с контроля",
    "Активный (срок окончания, предусмотренный Порядком не наступил, находится в стадии реализации)",
    "11_ККБ/МСБ_ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "5_ МСБ_Право Банка не реализовано, ФП снят с контроля",
    "6_ККБ/МСБ_Техническое закрытие ФП",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Активный (срок окончания, предусмотренный Порядком наступил, результат реализации по состоянию на дату Реестра отсутствует)",
    "Микро_У ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "8_ККБ/МСБ_В отношении ФП принято решение УОБ",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "ФП не требует устранения, право Банка не реализовано",
    "Установлена коммерческая ставка",
    "Микро_У Фактор устранен - решение УОБ о снятии фактора с контроля",
    "9_ККБ/МСБ_В отношении ФП принято решение УЛ",
    "ДКБ_Информация проверена, выявлен соответствующий ФП/СФП",
    "Микро_У Принятие решения УЛ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ/УЛ о снятии ФП с контроля",
    "По ФП реализовано право Банка",
]

# «Финальные» нейтральные результаты сценария
SCR_FINAL_NEUTRAL = [
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "5_ МСБ_Право Банка не реализовано, ФП снят с контроля",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "ФП не требует устранения, право Банка не реализовано",
    "Установлена коммерческая ставка",
    "По ФП реализовано право Банка",
]

# ── Классификация результатов ИПУ ──
IPU_POSITIVE = [
    "ФП устранен",
    "ФП устранен (договор закрыт)",
    "Принято решение УОБ о снятии с контроля",
    "Активный (продлен первоначальный срок окончания Плана, находится в стадии реализации)",
]
IPU_NEGATIVE = [
    "Активный (срок окончания Плана истек, результат реализации по состоянию на дату Реестра отсутствует)",
    "ФП не устранен, выявлены предпосылки, требующие рассмотрения вопроса о признании задолженности проблемной",
    "ФП не устранен в рамках Плана, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком (требует корректировки ИПУ)",
]
IPU_NEUTRAL = [
    "Активный (первоначально установленный срок окончания плана не наступил, находится в стадии реализации)",
]

_scr_sets = {
    "положительный": set(SCR_POSITIVE),
    "отрицательный": set(SCR_NEGATIVE),
    "нейтральный":   set(SCR_NEUTRAL),
    "ошибка":        set(SCR_ERROR),
}

_ipu_sets = {
    "положительный": set(IPU_POSITIVE),
    "отрицательный": set(IPU_NEGATIVE),
    "нейтральный":   set(IPU_NEUTRAL),
}


def classify_scr(val):
    for cls, s in _scr_sets.items():
        if val in s:
            return cls
    return "[пусто]" if val == "" else "неклассифицированный"


def classify_ipu(val):
    for cls, s in _ipu_sets.items():
        if val in s:
            return cls
    return "[пусто]" if val == "" else "неклассифицированный"


print(f"Сценарий: {len(SCR_POSITIVE)} полож., {len(SCR_NEGATIVE)} отриц., "
      f"{len(SCR_NEUTRAL)} нейтр., {len(SCR_ERROR)} ошибок = "
      f"{len(SCR_POSITIVE)+len(SCR_NEGATIVE)+len(SCR_NEUTRAL)+len(SCR_ERROR)}")
print(f"ИПУ: {len(IPU_POSITIVE)} полож., {len(IPU_NEGATIVE)} отриц., "
      f"{len(IPU_NEUTRAL)} нейтр. = "
      f"{len(IPU_POSITIVE)+len(IPU_NEGATIVE)+len(IPU_NEUTRAL)}")

In [ ]:
# Проверка: какие значения не попали ни в одну группу?
all_scr_classified = set(SCR_POSITIVE) | set(SCR_NEGATIVE) | set(SCR_NEUTRAL) | set(SCR_ERROR) | {""}
all_ipu_classified = set(IPU_POSITIVE) | set(IPU_NEGATIVE) | set(IPU_NEUTRAL) | {""}

scr_vals = set(raw["scenario"].unique())
ipu_vals = set(raw["ipu"].unique())

scr_unknown = scr_vals - all_scr_classified
ipu_unknown = ipu_vals - all_ipu_classified

if scr_unknown:
    print(f"⚠ Неклассифицированные результаты СЦЕНАРИЯ ({len(scr_unknown)}):")
    for v in sorted(scr_unknown):
        cnt = (raw["scenario"] == v).sum()
        print(f"  {cnt:>5,}  «{v}»")
else:
    print("✓ Все результаты сценария классифицированы")

print()

if ipu_unknown:
    print(f"⚠ Неклассифицированные результаты ИПУ ({len(ipu_unknown)}):")
    for v in sorted(ipu_unknown):
        cnt = (raw["ipu"] == v).sum()
        print(f"  {cnt:>5,}  «{v}»")
else:
    print("✓ Все результаты ИПУ классифицированы")

In [ ]:
# Базовые дедуплицированные наборы
first_df = raw.drop_duplicates(subset=DEDUP_KEY, keep="first")
last_df  = raw.drop_duplicates(subset=DEDUP_KEY, keep="last")
n_cards  = len(first_df)

print(f"keep=first: {len(first_df):,} карточек")
print(f"keep=last:  {len(last_df):,} карточек")


def compare_strategies(base_df, test_df, result_col, classify_fn, base_name, test_name):
    """Сравнивает два набора по колонке result_col, возвращает статистику переходов."""
    comp = base_df[DEDUP_KEY + [result_col]].merge(
        test_df[DEDUP_KEY + [result_col]],
        on=DEDUP_KEY, suffixes=("_base", "_test")
    )
    comp["cls_base"] = comp[f"{result_col}_base"].apply(classify_fn)
    comp["cls_test"] = comp[f"{result_col}_test"].apply(classify_fn)
    changed = comp[comp["cls_base"] != comp["cls_test"]]
    n = len(comp)
    nc = len(changed)
    pct = nc / max(n, 1) * 100
    print(f"\n{base_name} → {test_name}: {nc:,} из {n:,} ({pct:.1f}%) сменили класс")
    if nc > 0:
        trans = pd.crosstab(changed["cls_base"], changed["cls_test"],
                            margins=True, margins_name="Итого")
        trans.index.name = base_name
        trans.columns.name = test_name
        display(trans)
    return comp


def summarize_classes(dfs_dict, result_col, classify_fn):
    """Сводная таблица: распределение классов по стратегиям."""
    rows = []
    for name, df in dfs_dict.items():
        vc = df[result_col].apply(classify_fn).value_counts()
        row = {"Стратегия": name, "Всего": len(df)}
        for cls in ["положительный", "отрицательный", "нейтральный", "ошибка", "[пусто]", "неклассифицированный"]:
            row[cls] = vc.get(cls, 0)
        rows.append(row)
    summary = pd.DataFrame(rows)

    # Абсолютные
    print("Распределение классов (абсолютные):")
    display(summary.style.hide(axis="index"))

    # %
    pct = summary.copy()
    for cls in ["положительный", "отрицательный", "нейтральный", "ошибка", "[пусто]", "неклассифицированный"]:
        if cls in pct.columns:
            pct[cls] = (pct[cls] / pct["Всего"] * 100).round(1).astype(str) + "%"
    print("\nВ процентах:")
    display(pct.drop(columns=["Всего"]).style.hide(axis="index"))
    return summary

---

## Часть A. Стратегии для результата СЦЕНАРИЯ (`DESC_TEXT_1`)

Даты-маркеры: `END_DATE_SCR_FCT` (факт), `END_DATE_SCR_PLAN` (план)

### A1. Заполняемость дат сценария

In [ ]:
scr_dt_cols = [f"_{c}_dt" for c in SCR_DATE_COLS]

for col, dt_col in zip(SCR_DATE_COLS, scr_dt_cols):
    filled = raw[dt_col].notna().sum()
    print(f"{col}: {filled:,} из {len(raw):,} ({filled/len(raw)*100:.1f}%)")

raw["_scr_max_dt"] = raw[scr_dt_cols].max(axis=1)
has_scr_dt = raw["_scr_max_dt"].notna().sum()
print(f"\nmax(END_DATE_SCR_FCT, END_DATE_SCR_PLAN): {has_scr_dt:,} ({has_scr_dt/len(raw)*100:.1f}%)")

# Разброс внутри дубликатных групп
dup_mask = raw.duplicated(subset=DEDUP_KEY, keep=False)
dup_grp = raw[dup_mask].groupby(DEDUP_KEY)["_scr_max_dt"]
n_dup_groups = dup_grp.ngroups
diff_scr = (dup_grp.nunique() > 1).sum()
print(f"\nДубликатных групп: {n_dup_groups:,}")
print(f"Групп с разными max(дата сценария): {diff_scr:,} ({diff_scr/max(n_dup_groups,1)*100:.1f}%)")

### A2. Стратегия 3: max(END_DATE_SCR_FCT, END_DATE_SCR_PLAN)

In [ ]:
raw["_has_scr_dt"] = raw["_scr_max_dt"].notna()
scr_sorted = raw.sort_values(["_has_scr_dt", "_scr_max_dt"], ascending=[False, False])
scr_max_df = scr_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()

n_with = scr_max_df["_has_scr_dt"].sum()
print(f"Стратегия 3 (max дата сценария): {len(scr_max_df):,} карточек")
print(f"  с датой: {n_with:,}, fallback: {len(scr_max_df) - n_with:,}")

compare_strategies(first_df, scr_max_df, "scenario", classify_scr,
                   "keep=first", "max(дата сценария)")

### A3. Стратегия 4: closest(дата сценария → DATE_END_FP_SFP)

In [ ]:
scr_dist_cols = []
for col in SCR_DATE_COLS:
    dc = f"_dist_scr_{col}"
    raw[dc] = (raw[f"_{col}_dt"] - raw["_DATE_END_FP_SFP_dt"]).abs()
    scr_dist_cols.append(dc)

raw["_scr_min_dist"] = raw[scr_dist_cols].min(axis=1)
raw["_has_scr_dist"] = raw["_scr_min_dist"].notna()

scr_closest_sorted = raw.sort_values(["_has_scr_dist", "_scr_min_dist"], ascending=[False, True])
scr_closest_df = scr_closest_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()

n_with_dist = scr_closest_df["_has_scr_dist"].sum()
print(f"Стратегия 4 (closest сценарий → DATE_END_FP_SFP): {len(scr_closest_df):,} карточек")
print(f"  с расстоянием: {n_with_dist:,}, fallback: {len(scr_closest_df) - n_with_dist:,}")

compare_strategies(first_df, scr_closest_df, "scenario", classify_scr,
                   "keep=first", "closest(сценарий→DATE_END)")

### A4. Стратегия 5: только финальные результаты сценария (+/-)

In [ ]:
# 5a: только положительные + отрицательные
_scr_final = set(SCR_POSITIVE) | set(SCR_NEGATIVE)
scr_final_rows = raw[raw["scenario"].isin(_scr_final)]
scr_final_df = scr_final_rows.drop_duplicates(subset=DEDUP_KEY, keep="last")

n_lost_5a = n_cards - len(scr_final_df)
print(f"5a. Только полож. + отриц. результаты сценария:")
print(f"  Остаётся: {len(scr_final_df):,} | Потеряно: {n_lost_5a:,} ({n_lost_5a/n_cards*100:.1f}%)")

# 5b: + 8 «финальных» нейтральных
_scr_final_ext = _scr_final | set(SCR_FINAL_NEUTRAL)
scr_final_ext_rows = raw[raw["scenario"].isin(_scr_final_ext)]
scr_final_ext_df = scr_final_ext_rows.drop_duplicates(subset=DEDUP_KEY, keep="last")

n_lost_5b = n_cards - len(scr_final_ext_df)
print(f"\n5b. + 8 финальных нейтральных:")
print(f"  Остаётся: {len(scr_final_ext_df):,} | Потеряно: {n_lost_5b:,} ({n_lost_5b/n_cards*100:.1f}%)")
print(f"\n  8 нейтральных спасли {n_lost_5a - n_lost_5b:,} карточек")

# Какие результаты у потерянных в 5a?
lost_5a = first_df.merge(scr_final_df[DEDUP_KEY].drop_duplicates(),
                         on=DEDUP_KEY, how="left", indicator=True) \
                  .query("_merge == 'left_only'").drop(columns="_merge")
print(f"\nТоп-10 результатов потерянных карточек (5a):")
for val, cnt in lost_5a["scenario"].value_counts().head(10).items():
    print(f"  {cnt:>5,}  {val or '[пусто]'}")

### A5. Сводка по стратегиям сценария

In [ ]:
scr_strategies = {
    "1. keep=first (текущий)":           first_df,
    "2. keep=last":                      last_df,
    "3. max(дата сценария)":             scr_max_df,
    "4. closest(сценарий→DATE_END)":     scr_closest_df,
    "5a. только +/- результаты":         scr_final_df,
    "5b. +/- и 8 финальных нейтральных": scr_final_ext_df,
}

print("=" * 70)
print("СВОДКА: стратегии выбора записи для РЕЗУЛЬТАТА СЦЕНАРИЯ")
print("=" * 70)
scr_summary = summarize_classes(scr_strategies, "scenario", classify_scr)

---

## Часть B. Стратегии для результата ИПУ (`DESC_TEXT`)

Даты-маркеры: `FIRST_END_DATE_EVENT` (план), `NEW_PLAN_END_DATE_EVT` (новый план),
`END_EVENT_DATE_FACT` (факт)

ИПУ назначается только при негативных сценариях, поэтому многие карточки
не имеют заполненных ИПУ-колонок.

### B1. Заполняемость дат ИПУ

In [ ]:
ipu_dt_cols = [f"_{c}_dt" for c in IPU_DATE_COLS]

for col, dt_col in zip(IPU_DATE_COLS, ipu_dt_cols):
    filled = raw[dt_col].notna().sum()
    print(f"{col}: {filled:,} из {len(raw):,} ({filled/len(raw)*100:.1f}%)")

raw["_ipu_max_dt"] = raw[ipu_dt_cols].max(axis=1)
has_ipu_dt = raw["_ipu_max_dt"].notna().sum()
print(f"\nmax(IPU dates): {has_ipu_dt:,} ({has_ipu_dt/len(raw)*100:.1f}%)")

# Сколько карточек вообще имеют заполненный результат ИПУ?
has_ipu_result = (raw["ipu"] != "").sum()
print(f"Записей с заполненным DESC_TEXT (ИПУ): {has_ipu_result:,} ({has_ipu_result/len(raw)*100:.1f}%)")

# После дедупликации
ipu_first = (first_df["ipu"] != "").sum()
ipu_last  = (last_df["ipu"] != "").sum()
print(f"\nПосле дедупликации — карточки с заполненным ИПУ:")
print(f"  keep=first: {ipu_first:,} из {n_cards:,} ({ipu_first/n_cards*100:.1f}%)")
print(f"  keep=last:  {ipu_last:,} из {n_cards:,} ({ipu_last/n_cards*100:.1f}%)")

# Разброс внутри дубликатных групп
dup_ipu = raw[dup_mask].groupby(DEDUP_KEY)["_ipu_max_dt"]
diff_ipu = (dup_ipu.nunique() > 1).sum()
print(f"\nГрупп с разными max(дата ИПУ): {diff_ipu:,} из {n_dup_groups:,}")

### B2. Стратегия 3: max(FIRST_END_DATE_EVENT, NEW_PLAN_END_DATE_EVT, END_EVENT_DATE_FACT)

In [ ]:
raw["_has_ipu_dt"] = raw["_ipu_max_dt"].notna()
ipu_sorted = raw.sort_values(["_has_ipu_dt", "_ipu_max_dt"], ascending=[False, False])
ipu_max_df = ipu_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()

n_with_ipu = ipu_max_df["_has_ipu_dt"].sum()
print(f"Стратегия 3 (max дата ИПУ): {len(ipu_max_df):,} карточек")
print(f"  с датой ИПУ: {n_with_ipu:,}, fallback: {len(ipu_max_df) - n_with_ipu:,}")

compare_strategies(first_df, ipu_max_df, "ipu", classify_ipu,
                   "keep=first", "max(дата ИПУ)")

### B3. Стратегия 4: closest(дата ИПУ → DATE_END_FP_SFP)

In [ ]:
ipu_dist_cols = []
for col in IPU_DATE_COLS:
    dc = f"_dist_ipu_{col}"
    raw[dc] = (raw[f"_{col}_dt"] - raw["_DATE_END_FP_SFP_dt"]).abs()
    ipu_dist_cols.append(dc)

raw["_ipu_min_dist"] = raw[ipu_dist_cols].min(axis=1)
raw["_has_ipu_dist"] = raw["_ipu_min_dist"].notna()

ipu_closest_sorted = raw.sort_values(["_has_ipu_dist", "_ipu_min_dist"], ascending=[False, True])
ipu_closest_df = ipu_closest_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()

n_with_ipu_dist = ipu_closest_df["_has_ipu_dist"].sum()
print(f"Стратегия 4 (closest ИПУ → DATE_END_FP_SFP): {len(ipu_closest_df):,} карточек")
print(f"  с расстоянием: {n_with_ipu_dist:,}, fallback: {len(ipu_closest_df) - n_with_ipu_dist:,}")

compare_strategies(first_df, ipu_closest_df, "ipu", classify_ipu,
                   "keep=first", "closest(ИПУ→DATE_END)")

### B4. Стратегия 5: только финальные результаты ИПУ (+/-)

In [ ]:
# Для ИПУ: только записи с заполненным и финальным результатом (+/-)
_ipu_final = set(IPU_POSITIVE) | set(IPU_NEGATIVE)

ipu_final_rows = raw[raw["ipu"].isin(_ipu_final)]
ipu_final_df = ipu_final_rows.drop_duplicates(subset=DEDUP_KEY, keep="last")

# Базовая линия: сколько карточек вообще имеют непустой ИПУ
cards_with_ipu = raw[raw["ipu"] != ""].drop_duplicates(subset=DEDUP_KEY)
n_cards_with_ipu = len(cards_with_ipu)

n_lost_ipu = n_cards_with_ipu - len(ipu_final_df)

print(f"Всего карточек: {n_cards:,}")
print(f"Карточек с непустым ИПУ (хотя бы в одной записи): {n_cards_with_ipu:,}")
print(f"\nФильтр: только полож. + отриц. результаты ИПУ:")
print(f"  Остаётся: {len(ipu_final_df):,}")
print(f"  Потеряно от карточек с ИПУ: {n_lost_ipu:,} ({n_lost_ipu/max(n_cards_with_ipu,1)*100:.1f}%)")

# Какие результаты у потерянных?
lost_ipu = cards_with_ipu.merge(ipu_final_df[DEDUP_KEY].drop_duplicates(),
                                on=DEDUP_KEY, how="left", indicator=True) \
                         .query("_merge == 'left_only'").drop(columns="_merge")

if len(lost_ipu) > 0:
    print(f"\nРезультаты ИПУ у потерянных карточек:")
    for val, cnt in lost_ipu["ipu"].value_counts().items():
        print(f"  {cnt:>5,}  {val or '[пусто]'}")

### B5. Сводка по стратегиям ИПУ

In [ ]:
ipu_strategies = {
    "1. keep=first (текущий)":        first_df,
    "2. keep=last":                   last_df,
    "3. max(дата ИПУ)":              ipu_max_df,
    "4. closest(ИПУ→DATE_END)":      ipu_closest_df,
    "5. только +/- результаты ИПУ":  ipu_final_df,
}

print("=" * 70)
print("СВОДКА: стратегии выбора записи для РЕЗУЛЬТАТА ИПУ")
print("=" * 70)
ipu_summary = summarize_classes(ipu_strategies, "ipu", classify_ipu)

---

## Часть C. Перекрёстное сравнение и рекомендация

Какая стратегия лучше всего подходит для каждого этапа?
Дополнительно: проверяем согласованность стратегий
(одинаковая ли запись выбирается для сценария и ИПУ?).

In [ ]:
# Попарное сравнение стратегий сценария (кто больше финальных результатов даёт?)
print("=" * 70)
print("ПОПАРНОЕ СРАВНЕНИЕ СТРАТЕГИЙ СЦЕНАРИЯ")
print("=" * 70)

scr_pairs = [
    ("1. keep=first", first_df, "2. keep=last", last_df),
    ("1. keep=first", first_df, "3. max(дата сценария)", scr_max_df),
    ("1. keep=first", first_df, "4. closest(сцен→DATE_END)", scr_closest_df),
    ("2. keep=last", last_df, "3. max(дата сценария)", scr_max_df),
    ("3. max(дата сценария)", scr_max_df, "4. closest(сцен→DATE_END)", scr_closest_df),
]

for name_a, df_a, name_b, df_b in scr_pairs:
    compare_strategies(df_a, df_b, "scenario", classify_scr, name_a, name_b)

In [ ]:
# Попарное сравнение стратегий ИПУ
print("=" * 70)
print("ПОПАРНОЕ СРАВНЕНИЕ СТРАТЕГИЙ ИПУ")
print("=" * 70)

ipu_pairs = [
    ("1. keep=first", first_df, "2. keep=last", last_df),
    ("1. keep=first", first_df, "3. max(дата ИПУ)", ipu_max_df),
    ("1. keep=first", first_df, "4. closest(ИПУ→DATE_END)", ipu_closest_df),
    ("2. keep=last", last_df, "3. max(дата ИПУ)", ipu_max_df),
    ("3. max(дата ИПУ)", ipu_max_df, "4. closest(ИПУ→DATE_END)", ipu_closest_df),
]

for name_a, df_a, name_b, df_b in ipu_pairs:
    compare_strategies(df_a, df_b, "ipu", classify_ipu, name_a, name_b)

In [ ]:
# Согласованность: совпадает ли выбранная запись между стратегиями?
# Сравниваем индексы строк из raw для стратегий 3 (max дата) — сценарий vs ИПУ
scr_idx = set(scr_max_df.index)
ipu_idx = set(ipu_max_df.index)
overlap = scr_idx & ipu_idx
pct_overlap = len(overlap) / max(len(scr_idx), 1) * 100

print("Согласованность стратегии «max дата» между сценарием и ИПУ:")
print(f"  Одна и та же строка выбрана: {len(overlap):,} из {len(scr_idx):,} ({pct_overlap:.1f}%)")
print(f"  Разные строки: {len(scr_idx) - len(overlap):,}")

# То же для closest
scr_cl_idx = set(scr_closest_df.index)
ipu_cl_idx = set(ipu_closest_df.index)
overlap_cl = scr_cl_idx & ipu_cl_idx
pct_cl = len(overlap_cl) / max(len(scr_cl_idx), 1) * 100

print(f"\nСогласованность стратегии «closest → DATE_END» между сценарием и ИПУ:")
print(f"  Одна и та же строка: {len(overlap_cl):,} из {len(scr_cl_idx):,} ({pct_cl:.1f}%)")
print(f"  Разные строки: {len(scr_cl_idx) - len(overlap_cl):,}")

In [ ]:
print("=" * 70)
print("ИТОГ")
print("=" * 70)

# Подсчитываем долю «финальных» результатов (полож.+отриц.) для каждой стратегии
def final_pct(df, col, cls_fn):
    classes = df[col].apply(cls_fn)
    n_final = ((classes == "положительный") | (classes == "отрицательный")).sum()
    return n_final / max(len(df), 1) * 100

print("\n% карточек с финальным результатом (+/-):\n")
print("  СЦЕНАРИЙ:")
for name, df in scr_strategies.items():
    pct = final_pct(df, "scenario", classify_scr)
    bar = "#" * int(pct / 2)
    print(f"    {name:<42s} {pct:5.1f}%  {bar}")

print("\n  ИПУ:")
for name, df in ipu_strategies.items():
    pct = final_pct(df, "ipu", classify_ipu)
    bar = "#" * int(pct / 2)
    print(f"    {name:<42s} {pct:5.1f}%  {bar}")

print("\nПримечание: для стратегий 5/5a/5b процент будет 100% по определению,")
print("но размер выборки меньше (потеряны карточки без финального результата).")